In [1]:
import pandas as pd

In [2]:
 netflix_titles = pd.read_csv(
        '/kaggle/input/netflix-prize-data/movie_titles.csv',
        encoding='ISO-8859-1',
        header=None,
        names=['movieId', 'year', 'title'],
        on_bad_lines='skip',
        dtype={'movieId': 'int32', 'year': 'float32'} # Optimize types
    )
movie_genome_scores = pd.read_csv('/kaggle/input/movielens-20m-dataset/genome_scores.csv')
movie_genome_tags = pd.read_csv('/kaggle/input/movielens-20m-dataset/genome_tags.csv')
movies = pd.read_csv('/kaggle/input/movielens-20m-dataset/movie.csv')
movie_ratings = pd.read_csv('/kaggle/input/movielens-20m-dataset/rating.csv')
movie_tag = pd.read_csv('/kaggle/input/movielens-20m-dataset/tag.csv')

In [3]:
# List of your DataFrames
dataframes = {
    'netflix_titles': netflix_titles,
    'movie_genome_scores': movie_genome_scores,
    'movie_genome_tags': movie_genome_tags,
    'movies': movies,
    'movie_ratings': movie_ratings,
    'movie_tag': movie_tag
}

# Loop through and print info + sample rows
for name, df in dataframes.items():
    print(f"\n DataFrame: {name}")
    print(f" Number of Columns: {len(df.columns)}")
    print(f" Column Names: {df.columns.tolist()}")
    print(f" Head (first 5 rows):\n{df.head()}")
    print("-" * 80)



 DataFrame: netflix_titles
 Number of Columns: 3
 Column Names: ['movieId', 'year', 'title']
 Head (first 5 rows):
   movieId    year                         title
0        1  2003.0               Dinosaur Planet
1        2  2004.0    Isle of Man TT 2004 Review
2        3  1997.0                     Character
3        4  1994.0  Paula Abdul's Get Up & Dance
4        5  2004.0      The Rise and Fall of ECW
--------------------------------------------------------------------------------

 DataFrame: movie_genome_scores
 Number of Columns: 3
 Column Names: ['movieId', 'tagId', 'relevance']
 Head (first 5 rows):
   movieId  tagId  relevance
0        1      1    0.02500
1        1      2    0.02500
2        1      3    0.05775
3        1      4    0.09675
4        1      5    0.14675
--------------------------------------------------------------------------------

 DataFrame: movie_genome_tags
 Number of Columns: 2
 Column Names: ['tagId', 'tag']
 Head (first 5 rows):
   tagId           ta

In [8]:
import os
import gc
import pandas as pd
import numpy as np
from datetime import datetime
from scipy import sparse
import dask.dataframe as dd    # pip install dask[complete]

# ─── CONFIG ──────────────────────────────────────────────────────────
KAGGLE_NETFLIX_DIR = '/kaggle/input/netflix-prize-data/'
NETFLIX_FILE       = os.path.join(KAGGLE_NETFLIX_DIR, 'combined_data_1.txt')
OUTPUT_DIR         = '/kaggle/working/processed_data'
ML_RATINGS_CSV     = '/kaggle/input/movielens-20m-dataset/rating.csv'
ML_MOVIES_CSV      = '/kaggle/input/movielens-20m-dataset/movie.csv'
GENOME_SCORES_CSV  = '/kaggle/input/movielens-20m-dataset/genome_scores.csv'
# CORRECTED LINE BELOW:
GENOME_TAGS_CSV    = '/kaggle/input/movielens-20m-dataset/genome_tags.csv' # Changed from tag.csv

NETFLIX_CHUNK_SIZE = 500_000     # lines per chunk
# ────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ... (rest of your code remains the same)

def downcast_df(df):
    """ Downcast numeric columns to save memory """
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    return df

# ─── 1. Process MovieLens into Parquet ───────────────────────────────
print("Loading MovieLens…")
ml_ratings = pd.read_csv(ML_RATINGS_CSV)
ml_movies  = pd.read_csv(ML_MOVIES_CSV)
genome_s   = pd.read_csv(GENOME_SCORES_CSV)
genome_t   = pd.read_csv(GENOME_TAGS_CSV)

print("Processing MovieLens interactions…")
ml_ratings.rename(
    columns={'userId':'userId','movieId':'itemId','rating':'rating','timestamp':'timestamp'},
    inplace=True
)
# Parse timestamps: try epoch seconds, else fallback to string parse
def parse_timestamp(col):
    try:
        return pd.to_datetime(col, unit='s', errors='raise')
    except Exception:
        return pd.to_datetime(col, errors='coerce')

ml_ratings['timestamp'] = parse_timestamp(ml_ratings['timestamp'])
ml_ratings['source']    = 'MovieLens'
ml_ratings = downcast_df(ml_ratings)
ml_ratings.to_parquet(
    os.path.join(OUTPUT_DIR, 'ml_interactions.parquet'),
    index=False
)
del ml_ratings; gc.collect()

print("Processing MovieLens items + genome features…")
# base items
ml_items = ml_movies.rename(columns={'movieId':'itemId','title':'title'})
ml_items['year']        = ml_items['title'].str.extract(r'\((\d{4})\)').astype(float)
ml_items['type']        = 'movie'
ml_items['description'] = ''
ml_items = downcast_df(ml_items)

# genome pivot
genome = genome_s.merge(genome_t, on='tagId', how='left')
genome = genome.astype({'movieId':'int32','tagId':'int16','relevance':'float32'})
pivot = genome.pivot_table(
    index='movieId', columns='tag', values='relevance'
).fillna(0).add_prefix('tag_genome_').reset_index()
pivot.rename(columns={'movieId':'itemId'}, inplace=True)

ml_items = ml_items.merge(pivot, on='itemId', how='left').fillna(0)
for c in ml_items.columns:
    if c.startswith('tag_genome_'):
        ml_items[c] = ml_items[c].astype('float16')

ml_items.to_parquet(
    os.path.join(OUTPUT_DIR, 'ml_items.parquet'),
    index=False
)
del ml_movies, genome, pivot, genome_s, genome_t; gc.collect()

# ─── 2. Stream Netflix in chunks ───────────────────────────────────────
def parse_and_dump_netflix(in_path, chunk_size, out_dir):
    buf = []
    current_movie = None
    chunk_id = 0

    with open(in_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line.endswith(':'):
                current_movie = line[:-1]
            else:
                user_id, rating, date_str = line.split(',')
                buf.append({
                    'userId':   int(user_id),
                    'itemId':   f"netflix_{current_movie}",
                    'rating':   float(rating),
                    'timestamp': datetime.strptime(date_str, '%Y-%m-%d'),
                    'source':   'Netflix'
                })
            if len(buf) >= chunk_size:
                df = pd.DataFrame(buf)
                df = downcast_df(df)
                df.to_parquet(
                    os.path.join(out_dir, f'netflix_interactions_{chunk_id}.parquet'),
                    index=False
                )
                print(f"Flushed Netflix chunk {chunk_id} ({len(buf)} rows)")
                chunk_id += 1
                buf.clear()
                del df; gc.collect()

    # final remainder
    if buf:
        df = pd.DataFrame(buf)
        df = downcast_df(df)
        df.to_parquet(
            os.path.join(out_dir, f'netflix_interactions_{chunk_id}.parquet'),
            index=False
        )
        print(f"Flushed Netflix final chunk {chunk_id} ({len(buf)} rows)")
        del df; gc.collect()

print("Chunked parsing Netflix Prize data…")
parse_and_dump_netflix(NETFLIX_FILE, NETFLIX_CHUNK_SIZE, OUTPUT_DIR)

# ─── 3. Combine all interactions lazily with Dask ─────────────────────
print("Combining interactions via Dask…")
ml_int = dd.read_parquet(os.path.join(OUTPUT_DIR, 'ml_interactions.parquet'))
nf_int = dd.read_parquet(os.path.join(OUTPUT_DIR, 'netflix_interactions_*.parquet'))
all_int = dd.concat([ml_int, nf_int]).persist()

# Downcast & categories
all_int['userId']   = all_int['userId'].astype('int32').astype('category')
all_int['itemId']   = all_int['itemId'].astype('category')
all_int['rating']   = all_int['rating'].astype('float32')
all_int['timestamp']= dd.to_datetime(all_int['timestamp'])
all_int.to_parquet(os.path.join(OUTPUT_DIR, 'all_interactions.parquet'), overwrite=True)
del ml_int, nf_int, all_int; gc.collect()

# ─── 4. Build a sparse User–Item matrix ──────────────────────────────
print("Building sparse matrix…")
# Read back as pandas categories
ints = pd.read_parquet(os.path.join(OUTPUT_DIR, 'all_interactions.parquet'))
ints['user_cat'] = ints['userId'].cat.codes
ints['item_cat'] = ints['itemId'].cat.codes

row = ints['user_cat'].values
col = ints['item_cat'].values
data= ints['rating'].values

ui_sparse = sparse.csr_matrix(
    (data, (row, col)),
    shape=(ints['user_cat'].nunique(), ints['item_cat'].nunique())
)

print("Shape:", ui_sparse.shape)
print("Nonzero ratings:", ui_sparse.nnz)
print("Memory footprint (bytes):", ui_sparse.data.nbytes + ui_sparse.indptr.nbytes + ui_sparse.indices.nbytes)

print("\n✅ ETL complete. You now have:")
print(" - ./processed_data/ml_items.parquet")
print(" - ./processed_data/all_interactions.parquet")
print(" - A sparse CSR matrix `ui_sparse` ready for CF.")

# You can optionally delete `ints` and call gc.collect() to free RAM now.
del ints; gc.collect()


Loading MovieLens…
Processing MovieLens interactions…
Processing MovieLens items + genome features…
Chunked parsing Netflix Prize data…
Flushed Netflix chunk 0 (500000 rows)
Flushed Netflix chunk 1 (500000 rows)
Flushed Netflix chunk 2 (500000 rows)
Flushed Netflix chunk 3 (500000 rows)
Flushed Netflix chunk 4 (500000 rows)
Flushed Netflix chunk 5 (500000 rows)
Flushed Netflix chunk 6 (500000 rows)
Flushed Netflix chunk 7 (500000 rows)
Flushed Netflix chunk 8 (500000 rows)
Flushed Netflix chunk 9 (500000 rows)
Flushed Netflix chunk 10 (500000 rows)
Flushed Netflix chunk 11 (500000 rows)
Flushed Netflix chunk 12 (500000 rows)
Flushed Netflix chunk 13 (500000 rows)
Flushed Netflix chunk 14 (500000 rows)
Flushed Netflix chunk 15 (500000 rows)
Flushed Netflix chunk 16 (500000 rows)
Flushed Netflix chunk 17 (500000 rows)
Flushed Netflix chunk 18 (500000 rows)
Flushed Netflix chunk 19 (500000 rows)
Flushed Netflix chunk 20 (500000 rows)
Flushed Netflix chunk 21 (500000 rows)
Flushed Netflix 

ValueError: Failed to convert partition to expected pyarrow schema:
    `ArrowInvalid('Integer value 32113 not in range: -128 to 127', 'Conversion failed for column userId with type category')`

Expected partition schema:
    userId: dictionary<values=string, indices=int8, ordered=0>
    itemId: dictionary<values=string, indices=int8, ordered=0>
    rating: float
    timestamp: timestamp[ns]
    source: large_string
    __null_dask_index__: int64

Received partition schema:
    userId: dictionary<values=int32, indices=int32, ordered=0>
    itemId: dictionary<values=string, indices=int16, ordered=0>
    rating: float
    timestamp: timestamp[ns]
    source: large_string
    __null_dask_index__: int64

This error *may* be resolved by passing in schema information for
the mismatched column(s) using the `schema` keyword in `to_parquet`.

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
from tqdm.auto import tqdm # Import tqdm for progress bars

# --- 0. Data Paths on Kaggle (as provided by user) ---
# Assuming your notebook or environment has access to these paths
# If running locally, you'd need to download these and adjust paths
kaggle_data_dir_ml = '/kaggle/input/movielens-20m-dataset/'
kaggle_data_dir_netflix = '/kaggle/input/netflix-prize-data/'

# Output directory for Parquet files
output_dir = './processed_data/'
os.makedirs(output_dir, exist_ok=True) # Create directory if it doesn't exist

# MovieLens 20M
ml_ratings_path = os.path.join(kaggle_data_dir_ml, 'rating.csv')
ml_movies_path = os.path.join(kaggle_data_dir_ml, 'movie.csv')

# Netflix Prize
netflix_combined_data_1_path = os.path.join(kaggle_data_dir_netflix, 'combined_data_1.txt')
netflix_movie_titles_path = os.path.join(kaggle_data_dir_netflix, 'movie_titles.csv')

# --- 1. Define Unified Schema Structures ---
unified_users = []
unified_items = []
unified_interactions = []

# --- 2. Load and Process Each Dataset ---

# --- 2.1. MovieLens 20M ---
print("Processing MovieLens 20M data...")
try:
    print("Loading MovieLens ratings...")
    # Optimize dtypes during loading if possible for large columns
    ml_ratings = pd.read_csv(ml_ratings_path,
                             dtype={'userId': 'int32', 'movieId': 'int32', 'rating': 'float32'})
    print("Loading MovieLens movies...")
    ml_movies = pd.read_csv(ml_movies_path,
                            dtype={'movieId': 'int32'})

    # Interactions
    ml_interactions = ml_ratings.rename(columns={
        'userId': 'userId',
        'movieId': 'itemId',
        'rating': 'rating',
        'timestamp': 'timestamp'
    })
    ml_interactions['source'] = 'MovieLens'
    ml_interactions['timestamp'] = pd.to_datetime(ml_interactions['timestamp']) # Corrected: auto-infer format
    unified_interactions.append(ml_interactions)
    print(f"MovieLens interactions processed: {len(ml_interactions)} records.")

    # Items (Movies)
    ml_items = ml_movies.rename(columns={
        'movieId': 'itemId',
        'title': 'title',
        'genres': 'genre'
    })
    ml_items['type'] = 'movie'
    ml_items['year'] = ml_items['title'].str.extract(r'\((\d{4})\)').astype(float)
    ml_items['description'] = None
    unified_items.append(ml_items[['itemId', 'title', 'type', 'genre', 'year', 'description']])
    print(f"MovieLens items processed: {len(ml_items)} records.")

    # Users - MovieLens only provides IDs
    ml_users = pd.DataFrame({'userId': ml_ratings['userId'].unique()})
    ml_users['source'] = 'MovieLens'
    unified_users.append(ml_users)
    print(f"MovieLens users processed: {len(ml_users)} records.")

except FileNotFoundError as e:
    print(f"MovieLens files not found: {e}. Ensure paths are correct: {ml_ratings_path}, {ml_movies_path}")
except Exception as e:
    print(f"Error processing MovieLens data: {e}")

# --- 2.2. Netflix Prize Dataset ---
print("\nProcessing Netflix Prize data (sample from combined_data_1.txt)...")
try:
    print("Loading Netflix movie titles...")
    netflix_titles = pd.read_csv(
        netflix_movie_titles_path,
        encoding='ISO-8859-1',
        header=None,
        names=['movieId', 'year', 'title'],
        on_bad_lines='skip',
        dtype={'movieId': 'int32', 'year': 'float32'} # Optimize types
    )
    netflix_titles['movieId'] = netflix_titles['movieId'].astype(str)
    print(f"Netflix movie titles loaded: {len(netflix_titles)} records.")

    print("Parsing Netflix combined_data_1.txt for interactions (this might take a while)...")
    netflix_raw_data = []
    with open(netflix_combined_data_1_path, 'r') as f:
        total_lines = sum(1 for _ in open(netflix_combined_data_1_path, 'r'))
        f.seek(0)
        current_movie_id_raw = None
        for line in tqdm(f, total=total_lines, desc="Reading Netflix Data"):
            line = line.strip()
            if line.endswith(':'):
                current_movie_id_raw = line[:-1]
            else:
                user_id, rating, date_str = line.split(',')
                timestamp = datetime.strptime(date_str, '%Y-%m-%d')
                netflix_raw_data.append({
                    'userId': int(user_id),
                    'itemId': f"netflix_{current_movie_id_raw}",
                    'rating': float(rating),
                    'timestamp': timestamp,
                    'source': 'Netflix'
                })
    netflix_interactions = pd.DataFrame(netflix_raw_data)
    # Further optimize types after DataFrame creation
    netflix_interactions['userId'] = netflix_interactions['userId'].astype('int32')
    netflix_interactions['rating'] = netflix_interactions['rating'].astype('float32')
    unified_interactions.append(netflix_interactions[['userId', 'itemId', 'rating', 'timestamp', 'source']])
    print(f"Netflix interactions processed: {len(netflix_interactions)} records.")

    # Items (Movies) for Netflix
    netflix_titles['unified_itemId'] = 'netflix_' + netflix_titles['movieId'].astype(str)
    netflix_items = netflix_titles.rename(columns={'title': 'title', 'year': 'year'})
    netflix_items['itemId'] = netflix_items['unified_itemId']
    netflix_items['type'] = 'movie'
    netflix_items['genre'] = 'Unknown'
    netflix_items['description'] = None
    unified_items.append(netflix_items[['itemId', 'title', 'type', 'genre', 'year', 'description']].drop_duplicates(subset=['itemId']))
    print(f"Netflix items processed: {len(netflix_items)} records.")

    # Users - Netflix also primarily provides IDs
    netflix_users = pd.DataFrame({'userId': netflix_interactions['userId'].unique()})
    netflix_users['source'] = 'Netflix'
    unified_users.append(netflix_users)
    print(f"Netflix users processed: {len(netflix_users)} records.")

except FileNotFoundError as e:
    print(f"Netflix Prize files not found: {e}. Ensure paths are correct: {netflix_combined_data_1_path}, {netflix_movie_titles_path}")
except Exception as e:
    print(f"Error processing Netflix data: {e}")

# --- 3. Consolidate Unified DataFrames ---
print("\nConsolidating unified dataframes...")

if unified_interactions:
    print("Concatenating unified interactions...")
    final_interactions_df = pd.concat(unified_interactions, ignore_index=True)
    final_interactions_df['userId'] = final_interactions_df['userId'].astype('str') # Ensure consistent string ID
    final_interactions_df['itemId'] = final_interactions_df['itemId'].astype('str') # Ensure consistent string ID
    final_interactions_df['source'] = final_interactions_df['source'].astype('category') # Optimize memory
    print(f"Total Unified Interactions: {len(final_interactions_df)} records")
    print(f"Memory usage after consolidation (interactions): {final_interactions_df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
    print(final_interactions_df.head())
else:
    final_interactions_df = pd.DataFrame()
    print("No interactions data loaded.")

if unified_items:
    print("\nConcatenating unified items...")
    final_items_df = pd.concat(unified_items, ignore_index=True)
    final_items_df['itemId'] = final_items_df['itemId'].astype('str')
    final_items_df.drop_duplicates(subset=['itemId'], inplace=True)
    final_items_df['type'] = final_items_df['type'].astype('category') # Optimize memory
    print(f"\nTotal Unified Items: {len(final_items_df)} unique records")
    print(f"Memory usage after consolidation (items): {final_items_df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
    print(final_items_df.head())
else:
    final_items_df = pd.DataFrame()
    print("No items data loaded.")

if unified_users:
    print("\nConcatenating unified users...")
    final_users_df = pd.concat(unified_users, ignore_index=True)
    final_users_df['userId'] = final_users_df['userId'].astype('str')
    final_users_df.drop_duplicates(subset=['userId'], inplace=True)
    final_users_df['source'] = final_users_df['source'].astype('category') # Optimize memory
    print(f"\nTotal Unified Users: {len(final_users_df)} unique records")
    print(f"Memory usage after consolidation (users): {final_users_df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
    print(final_users_df.head())
else:
    final_users_df = pd.DataFrame()
    print("No users data loaded.")

print("\n--- Phase 1: Data Integration Complete. Proceeding to Preprocessing & Feature Engineering ---")

# --- 4. Data Preprocessing & Initial Feature Engineering ---

# --- 4.1. Handle Missing Values in final_items_df ---
print("\nHandling missing values in final_items_df...")
final_items_df['year'].fillna(2000, inplace=True)
final_items_df['year'] = final_items_df['year'].astype('int16') # Optimize year type
final_items_df['genre'].fillna('Unknown', inplace=True)
final_items_df['description'].fillna('', inplace=True)
print("Missing values handled in final_items_df.")
print(f"Memory usage after missing value handling (items): {final_items_df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")


# --- 4.2. Process Genres ---
print("\nProcessing genres into lists and creating dummy variables...")
def split_genres(genres_string):
    if isinstance(genres_string, str):
        return [g.strip() for g in genres_string.split('|') if g.strip()]
    return []

tqdm.pandas(desc="Splitting genres")
final_items_df['genres_list'] = final_items_df['genre'].progress_apply(split_genres)

all_genres = set()
for genres_list in tqdm(final_items_df['genres_list'], desc="Collecting unique genres"):
    all_genres.update(genres_list)
all_genres = sorted(list(all_genres))

print("Creating binary genre features (One-Hot Encoding)...")
for genre in tqdm(all_genres, desc="Generating genre columns"):
    final_items_df[f'genre_{genre}'] = final_items_df['genres_list'].apply(lambda x: 1 if genre in x else 0).astype('int8') # Optimize to int8

final_items_df.drop(columns=['genre', 'genres_list'], inplace=True)
print("Genre processing complete.")
print(f"Memory usage after genre processing (items): {final_items_df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")


# --- 4.3. Basic User Features ---
print("\nExtracting basic user features...")
user_stats = final_interactions_df.groupby('userId')['rating'].agg(
    avg_rating=('mean'),
    num_ratings=('count')
).reset_index()
user_stats['avg_rating'] = user_stats['avg_rating'].astype('float32') # Optimize
user_stats['num_ratings'] = user_stats['num_ratings'].astype('int32') # Optimize

final_users_df = final_users_df.merge(user_stats, on='userId', how='left')
final_users_df['avg_rating'].fillna(0, inplace=True)
final_users_df['num_ratings'].fillna(0, inplace=True)
print("Basic user features extracted.")
print(f"Memory usage after basic user features (users): {final_users_df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")


# --- 4.4. Temporal Features from Interactions ---
print("\nExtracting temporal features from interactions...")
final_interactions_df['timestamp'] = pd.to_datetime(final_interactions_df['timestamp']) # Ensure datetime
tqdm.pandas(desc="Extracting temporal features")
final_interactions_df['interaction_year'] = final_interactions_df['timestamp'].dt.year.astype('int16')
final_interactions_df['interaction_month'] = final_interactions_df['timestamp'].dt.month.astype('int8')
final_interactions_df['interaction_dayofweek'] = final_interactions_df['timestamp'].dt.dayofweek.astype('int8')
final_interactions_df['interaction_hour'] = final_interactions_df['timestamp'].dt.hour.astype('int8')
final_interactions_df['is_weekend'] = final_interactions_df['interaction_dayofweek'].isin([5, 6]).astype('int8')
print("Temporal features extracted.")
print(f"Memory usage after temporal features (interactions): {final_interactions_df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")


print("\n--- Phase 1: Data Preprocessing and Initial Feature Engineering Complete ---")


# --- 5. More Advanced User Profiling ---
print("\nBuilding user genre preferences...")
genre_cols = [col for col in final_items_df.columns if col.startswith('genre_')]
items_with_genres = final_items_df[['itemId'] + genre_cols]

relevant_interactions = final_interactions_df[final_interactions_df['rating'] > 0].copy()
user_item_genre_interactions = relevant_interactions.merge(items_with_genres, on='itemId', how='left')
user_item_genre_interactions.dropna(subset=genre_cols, inplace=True)

for col in tqdm(genre_cols, desc="Weighting genres by rating"):
    user_item_genre_interactions[col] = user_item_genre_interactions[col] * user_item_genre_interactions['rating']

print("Grouping by userId to sum weighted genre preferences...")
user_genre_preferences = user_item_genre_interactions.groupby('userId')[genre_cols].sum()

tqdm.pandas(desc="Normalizing user genre preferences")
user_genre_preferences = user_genre_preferences.apply(lambda x: x / x.sum() if x.sum() != 0 else x, axis=1).fillna(0)
# Ensure resulting preferences are float32
for col in user_genre_preferences.columns:
    user_genre_preferences[col] = user_genre_preferences[col].astype('float32')

print("Merging user genre preferences into final_users_df...")
final_users_df = final_users_df.merge(user_genre_preferences, on='userId', how='left')
for col in tqdm(genre_cols, desc="Filling missing genre preferences"):
    final_users_df[col].fillna(0, inplace=True)

print("User genre preferences built and merged into final_users_df.")
print(f"Memory usage after advanced user profiling (users): {final_users_df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")


# --- Save DataFrames to Parquet ---
print("\n--- Saving optimized DataFrames to Parquet files ---")
final_interactions_df.to_parquet(os.path.join(output_dir, 'final_interactions.parquet'))
final_items_df.to_parquet(os.path.join(output_dir, 'final_items.parquet'))
final_users_df.to_parquet(os.path.join(output_dir, 'final_users.parquet'))
print(f"DataFrames saved to '{output_dir}'")

print("\nAll Phase 1 processing and saving complete. You can now use 'final_interactions.parquet', 'final_items.parquet', and 'final_users.parquet' directly in subsequent steps.")
